# 08. Признаки и target

Validation подробно показывает user, item, user-item и category features. Train/test повторяют эти уже объяснённые блоки без цикла и без обучения моделей.

## Подключение проекта

**Что делаем:** определяем корень проекта.  
**Зачем:** одинаковые пути должны работать локально и в Colab.  
**Что получим:** `PROJECT_ROOT` и доступный пакет из `src`.

In [1]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/fashion-recommender-system")
except ImportError:
    PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(f"Не найдена папка src: {PROJECT_ROOT / 'src'}")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Корень проекта:", PROJECT_ROOT)

Корень проекта: <PROJECT_ROOT>


### Зависимости

**Что делаем:** устанавливаем requirements только в Colab.  
**Зачем:** локальное окружение не должно изменяться при каждом запуске.  
**Что получим:** готовые библиотеки для следующих ячеек.

In [2]:
import subprocess

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(PROJECT_ROOT / "requirements.txt")],
        check=True,
    )

### Импорты

**Что делаем:** подключаем pandas, загрузчики и Candidate Recall  
**Зачем:** в feature notebook нет ALS, encoder или CatBoost  
**Что получим:** минимальный набор для агрегаций

In [3]:
import numpy as np
import pandas as pd
from IPython.display import display

from fashion_recommender.data import load_articles, load_customers, load_transactions
from fashion_recommender.evaluation import candidate_recall_at_k
from fashion_recommender.persistence import load_json

### Пути

**Что делаем:** задаём общие каталоги и проверяем temporal windows  
**Зачем:** настройка путей отделена от проверки candidate files  
**Что получим:** базовые пути

In [4]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports" / "tables"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
TRANSACTIONS_PATH = RAW_DIR / "transactions_train.csv"
ARTICLES_PATH = RAW_DIR / "articles.csv"
CUSTOMERS_PATH = RAW_DIR / "customers.csv"
for directory in [PROCESSED_DIR, MODEL_DIR, REPORT_DIR, ARTIFACT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

WINDOWS_PATH = PROCESSED_DIR / "temporal_windows.json"
if not WINDOWS_PATH.is_file():
    raise FileNotFoundError(
        f"Не найден файл: {WINDOWS_PATH}\n"
        "Сначала выполните notebook 03_temporal_validation_colab.ipynb."
    )

### Проверка merged candidates

**Что делаем:** проверяем три Parquet от notebook 07  
**Зачем:** feature engineering не должен запускать candidate generation  
**Что получим:** явные artifact contracts

In [5]:
CANDIDATE_PATHS = {
    split: PROCESSED_DIR / f"merged_candidates_{split}.parquet"
    for split in ["train", "validation", "test"]
}
missing_paths = [
    path for path in CANDIDATE_PATHS.values() if not path.is_file()
]
if missing_paths:
    raise FileNotFoundError(
        f"Не найдены merged candidates: {missing_paths}. "
        "Сначала выполните notebook 07_candidate_generation_colab.ipynb."
    )

### Категории

**Что делаем:** задаём item attributes и четыре affinity categories  
**Зачем:** названия должны совпасть с notebook 09  
**Что получим:** два понятных списка

In [6]:
ITEM_FEATURE_COLUMNS = [
    "product_type_name", "product_group_name", "colour_group_name",
    "department_name", "section_name", "garment_group_name",
]
AFFINITY_CATEGORIES = [
    "product_type_name",
    "colour_group_name",
    "section_name",
    "garment_group_name",
]
print("Item categories:", ITEM_FEATURE_COLUMNS)

Item categories: ['product_type_name', 'product_group_name', 'colour_group_name', 'department_name', 'section_name', 'garment_group_name']


### Загрузка данных

**Что делаем:** читаем исходные таблицы и windows один раз  
**Зачем:** каждый split дальше использует свой cutoff  
**Что получим:** четыре входных объекта

In [7]:
transactions = load_transactions(TRANSACTIONS_PATH)
articles = load_articles(ARTICLES_PATH)
customers = load_customers(CUSTOMERS_PATH)
windows = load_json(WINDOWS_PATH)
print("Transactions:", transactions.shape)
print("Articles:", articles.shape)
print("Customers:", customers.shape)

Transactions: (1048575, 5)
Articles: (105542, 25)
Customers: (1048575, 6)


### Article attributes

**Что делаем:** готовим только ID и шесть категорий  
**Зачем:** эти столбцы присоединяются к candidate pair  
**Что получим:** `article_attributes`

In [8]:
article_attributes = articles[
    ["article_id", *ITEM_FEATURE_COLUMNS]
].drop_duplicates("article_id").copy()
article_attributes[ITEM_FEATURE_COLUMNS] = (
    article_attributes[ITEM_FEATURE_COLUMNS]
    .fillna("Unknown")
    .astype(str)
)
display(article_attributes.head())

   article_id product_type_name  ...            section_name garment_group_name
0  0108775015          Vest top  ...  Womens Everyday Basics       Jersey Basic
1  0108775044          Vest top  ...  Womens Everyday Basics       Jersey Basic
2  0108775051          Vest top  ...  Womens Everyday Basics       Jersey Basic
3  0110065001               Bra  ...         Womens Lingerie  Under-, Nightwear
4  0110065002               Bra  ...         Womens Lingerie  Under-, Nightwear

[5 rows x 7 columns]


### Validation inputs

**Что делаем:** выбираем validation history/future и загружаем candidates  
**Зачем:** на одном окне подробно строятся все признаки  
**Что получим:** три validation tables

In [9]:
validation_cutoff = pd.Timestamp(windows["validation"]["cutoff_date"])
validation_end = pd.Timestamp(windows["validation"]["target_end_date"])
validation_history = transactions[
    transactions["t_dat"] < validation_cutoff
].copy()
validation_future = transactions[
    transactions["t_dat"].between(validation_cutoff, validation_end)
].copy()
validation_candidates = pd.read_parquet(
    CANDIDATE_PATHS["validation"]
)
assert validation_history["t_dat"].max() < validation_cutoff
print("History:", validation_history.shape)
print("Future:", validation_future.shape)
print("Candidates:", validation_candidates.shape)

History: (1011880, 5)
Future: (23405, 5)
Candidates: (444853, 15)


### User count features

**Что делаем:** считаем покупки, unique items и active days  
**Зачем:** это базовые признаки активности клиента  
**Что получим:** `validation_user_features`

In [10]:
validation_user_features = validation_history.groupby(
    "customer_id", as_index=False
).agg(
    user_total_purchases=("article_id", "size"),
    user_unique_items=("article_id", "nunique"),
    user_active_days=("t_dat", "nunique"),
)
display(validation_user_features.head())

                                         customer_id  ...  user_active_days
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...  ...                 2
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...  ...                 5
2  000064249685c11552da43ef22a5030f35a147f723d5b0...  ...                 1
3  0000757967448a6cb83efb3ea7a3fb9d418ac7adf2379d...  ...                 1
4  00007d2de826758b65a93dd24ce629ed66842531df6699...  ...                 5

[5 rows x 4 columns]


### Последняя покупка user

**Что делаем:** находим max date отдельно  
**Зачем:** из неё рассчитывается user recency  
**Что получим:** `validation_user_last_purchase`

In [11]:
validation_user_last_purchase = validation_history.groupby(
    "customer_id", as_index=False
)["t_dat"].max().rename(columns={"t_dat": "user_last_purchase"})
display(validation_user_last_purchase.head())

                                         customer_id user_last_purchase
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...         2019-09-28
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...         2019-08-12
2  000064249685c11552da43ef22a5030f35a147f723d5b0...         2019-10-02
3  0000757967448a6cb83efb3ea7a3fb9d418ac7adf2379d...         2019-06-04
4  00007d2de826758b65a93dd24ce629ed66842531df6699...         2019-11-24


### User recency

**Что делаем:** присоединяем дату и считаем дни до cutoff  
**Зачем:** future не участвует в признаке  
**Что получим:** `user_days_since_last_purchase`

In [12]:
validation_user_features = validation_user_features.merge(
    validation_user_last_purchase,
    on="customer_id",
    how="left",
)
validation_user_features["user_days_since_last_purchase"] = (
    validation_cutoff - validation_user_features.pop("user_last_purchase")
).dt.days
display(validation_user_features.head())

                                         customer_id  ...  user_days_since_last_purchase
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...  ...                             81
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...  ...                            128
2  000064249685c11552da43ef22a5030f35a147f723d5b0...  ...                             77
3  0000757967448a6cb83efb3ea7a3fb9d418ac7adf2379d...  ...                            197
4  00007d2de826758b65a93dd24ce629ed66842531df6699...  ...                             24

[5 rows x 5 columns]


### Средняя цена user

**Что делаем:** считаем mean price покупок  
**Зачем:** признак описывает привычный ценовой уровень  
**Что получим:** `user_average_price`

In [13]:
validation_user_price = validation_history.groupby(
    "customer_id", as_index=False
)["price"].mean().rename(columns={"price": "user_average_price"})
validation_user_features = validation_user_features.merge(
    validation_user_price,
    on="customer_id",
    how="left",
)
display(validation_user_features.head())

                                         customer_id  ...  user_average_price
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...  ...            0.052525
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...  ...            0.033316
2  000064249685c11552da43ef22a5030f35a147f723d5b0...  ...            0.042356
3  0000757967448a6cb83efb3ea7a3fb9d418ac7adf2379d...  ...            0.025407
4  00007d2de826758b65a93dd24ce629ed66842531df6699...  ...            0.046291

[5 rows x 6 columns]


### Online share user

**Что делаем:** помечаем канал 2 и считаем среднее  
**Зачем:** получаем долю online-покупок  
**Что получим:** `user_online_share`

In [14]:
validation_online_share = (
    validation_history
    .assign(_online=validation_history["sales_channel_id"].eq(2).astype(float))
    .groupby("customer_id", as_index=False)["_online"]
    .mean()
    .rename(columns={"_online": "user_online_share"})
)
validation_user_features = validation_user_features.merge(
    validation_online_share,
    on="customer_id",
    how="left",
)

### Возраст user

**Что делаем:** присоединяем age из customer profile  
**Зачем:** статичный профиль не агрегируется из future  
**Что получим:** `user_age` в user features

In [15]:
validation_age = customers[
    ["customer_id", "age"]
].drop_duplicates("customer_id").rename(columns={"age": "user_age"})
validation_user_features = validation_user_features.merge(
    validation_age,
    on="customer_id",
    how="left",
)
display(validation_user_features.head())

                                         customer_id  ...  user_age
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...  ...      49.0
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...  ...      25.0
2  000064249685c11552da43ef22a5030f35a147f723d5b0...  ...       NaN
3  0000757967448a6cb83efb3ea7a3fb9d418ac7adf2379d...  ...      20.0
4  00007d2de826758b65a93dd24ce629ed66842531df6699...  ...      32.0

[5 rows x 8 columns]


### Item count features

**Что делаем:** считаем покупки и уникальных покупателей товара  
**Зачем:** это базовая item popularity  
**Что получим:** `validation_item_features`

In [16]:
validation_item_features = validation_history.groupby(
    "article_id", as_index=False
).agg(
    item_total_purchases=("customer_id", "size"),
    item_unique_customers=("customer_id", "nunique"),
)
display(validation_item_features.head())

   article_id  item_total_purchases  item_unique_customers
0  0108775015                   369                    358
1  0108775044                   335                    328
2  0110065001                    42                     42
3  0110065002                    14                     13
4  0110065011                    32                     32


### Item price

**Что делаем:** считаем среднюю историческую цену  
**Зачем:** товары разных ценовых уровней различаются  
**Что получим:** `item_average_price`

In [17]:
validation_item_price = validation_history.groupby(
    "article_id", as_index=False
)["price"].mean().rename(columns={"price": "item_average_price"})
validation_item_features = validation_item_features.merge(
    validation_item_price,
    on="article_id",
    how="left",
)

### Последняя покупка item

**Что делаем:** находим max date каждого товара  
**Зачем:** item recency рассчитывается относительно cutoff  
**Что получим:** `item_days_since_last_purchase`

In [18]:
validation_item_last = validation_history.groupby(
    "article_id", as_index=False
)["t_dat"].max().rename(columns={"t_dat": "item_last_purchase"})
validation_item_features = validation_item_features.merge(
    validation_item_last,
    on="article_id",
    how="left",
)
validation_item_features["item_days_since_last_purchase"] = (
    validation_cutoff - validation_item_features.pop("item_last_purchase")
).dt.days

### Популярность item за 7 дней

**Что делаем:** считаем покупки в коротком history-окне  
**Зачем:** признак отражает свежий спрос  
**Что получим:** `item_popularity_7d`

In [19]:
validation_recent_7d = validation_history[
    validation_history["t_dat"] >= validation_cutoff - pd.Timedelta(days=7)
]
validation_popularity_7d = validation_recent_7d.groupby(
    "article_id", as_index=False
).size().rename(columns={"size": "item_popularity_7d"})
validation_item_features = validation_item_features.merge(
    validation_popularity_7d,
    on="article_id",
    how="left",
)

### Популярность item за 30 дней

**Что делаем:** считаем более устойчивый месячный спрос  
**Зачем:** 7d и 30d дают разные временные масштабы  
**Что получим:** `item_popularity_30d`

In [20]:
validation_recent_30d = validation_history[
    validation_history["t_dat"] >= validation_cutoff - pd.Timedelta(days=30)
]
validation_popularity_30d = validation_recent_30d.groupby(
    "article_id", as_index=False
).size().rename(columns={"size": "item_popularity_30d"})
validation_item_features = validation_item_features.merge(
    validation_popularity_30d,
    on="article_id",
    how="left",
)
display(validation_item_features.head())

   article_id  item_total_purchases  ...  item_popularity_7d  item_popularity_30d
0  0108775015                   369  ...                 NaN                  1.0
1  0108775044                   335  ...                 NaN                  1.0
2  0110065001                    42  ...                 NaN                  1.0
3  0110065002                    14  ...                 NaN                  NaN
4  0110065011                    32  ...                 NaN                  NaN

[5 rows x 7 columns]


### User-item count

**Что делаем:** считаем число прошлых покупок каждой пары  
**Зачем:** так модель узнаёт повторную покупку  
**Что получим:** `validation_pair_features`

In [21]:
validation_pair_features = validation_history.groupby(
    ["customer_id", "article_id"], as_index=False
).size().rename(columns={"size": "user_item_purchase_count"})
validation_pair_features["user_bought_item_before"] = 1
display(validation_pair_features.head())

                                         customer_id  ... user_bought_item_before
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...  ...                       1
1  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...  ...                       1
2  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...  ...                       1
3  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...  ...                       1
4  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...  ...                       1

[5 rows x 4 columns]


### User-item last purchase

**Что делаем:** находим последнюю дату пары  
**Зачем:** pair recency отличается от общей user recency  
**Что получим:** `days_since_user_bought_item`

In [22]:
validation_pair_last = validation_history.groupby(
    ["customer_id", "article_id"], as_index=False
)["t_dat"].max().rename(columns={"t_dat": "user_item_last_purchase"})
validation_pair_features = validation_pair_features.merge(
    validation_pair_last,
    on=["customer_id", "article_id"],
    how="left",
)
validation_pair_features["days_since_user_bought_item"] = (
    validation_cutoff - validation_pair_features.pop("user_item_last_purchase")
).dt.days
display(validation_pair_features.head())

                                         customer_id  ... days_since_user_bought_item
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...  ...                         207
1  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...  ...                          81
2  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...  ...                         128
3  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...  ...                         210
4  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...  ...                         183

[5 rows x 5 columns]


### History с категориями

**Что делаем:** добавляем item attributes к прошлым покупкам  
**Зачем:** affinity считается только из history  
**Что получим:** `validation_history_categories`

In [23]:
validation_history_categories = validation_history[
    ["customer_id", "article_id"]
].merge(
    article_attributes,
    on="article_id",
    how="left",
)
display(validation_history_categories.head())

                                         customer_id  ... garment_group_name
0  3e2b60b679e62fb49516105b975560082922011dd752ec...  ...       Jersey Fancy
1  89647ac2274f54c770aaa4b326e0eea09610c252381f37...  ...       Jersey Fancy
2  2ebe392150feb60ca89caa8eff6c08b7ef1138cd6fdc71...  ...             Shorts
3  7b3205de4ca17a339624eb5e3086698e9984eba6b47c56...  ...        Accessories
4  3b77905de8b32045f08cedb79200cdfa477e9562429a39...  ...   Socks and Tights

[5 rows x 8 columns]


### Пример Product Type affinity

**Что делаем:** считаем покупки user в каждом product type  
**Зачем:** этот пример объясняет остальные category counts  
**Что получим:** `validation_product_affinity`

In [24]:
validation_product_affinity = validation_history_categories.groupby(
    ["customer_id", "product_type_name"], as_index=False
).size().rename(columns={"size": "user_product_type_count"})
display(validation_product_affinity.head())

                                         customer_id  ... user_product_type_count
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...  ...                       2
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...  ...                       2
2  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...  ...                       1
3  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...  ...                       1
4  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...  ...                       2

[5 rows x 3 columns]


### Остальные affinity

**Что делаем:** повторяем тот же groupby для colour, section и garment  
**Зачем:** логика уже показана на product type  
**Что получим:** три небольшие таблицы

In [25]:
validation_colour_affinity = validation_history_categories.groupby(
    ["customer_id", "colour_group_name"], as_index=False
).size().rename(columns={"size": "user_colour_count"})
validation_section_affinity = validation_history_categories.groupby(
    ["customer_id", "section_name"], as_index=False
).size().rename(columns={"size": "user_section_count"})
validation_garment_affinity = validation_history_categories.groupby(
    ["customer_id", "garment_group_name"], as_index=False
).size().rename(columns={"size": "user_garment_group_count"})
display(validation_colour_affinity.head())

                                         customer_id  ... user_colour_count
0  00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...  ...                 2
1  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...  ...                 2
2  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...  ...                 1
3  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...  ...                 1
4  0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...  ...                 1

[5 rows x 3 columns]


### Merge user и item features

**Что делаем:** присоединяем два агрегатных блока к candidates  
**Зачем:** число candidate pairs не должно измениться  
**Что получим:** начало `validation_ranking_table`

In [26]:
validation_ranking_table = validation_candidates.merge(
    validation_user_features,
    on="customer_id",
    how="left",
)
validation_ranking_table = validation_ranking_table.merge(
    validation_item_features,
    on="article_id",
    how="left",
)
print("Ranking rows:", len(validation_ranking_table))

Ranking rows: 444853


### Merge attributes и pair features

**Что делаем:** добавляем категории товара и историю пары  
**Зачем:** каждый merge имеет понятный ключ  
**Что получим:** расширенную feature table

In [27]:
validation_ranking_table = validation_ranking_table.merge(
    article_attributes,
    on="article_id",
    how="left",
)
validation_ranking_table = validation_ranking_table.merge(
    validation_pair_features,
    on=["customer_id", "article_id"],
    how="left",
)
print("Columns:", len(validation_ranking_table.columns))

Columns: 37


### Merge affinity features

**Что делаем:** присоединяем четыре category counts по соответствующему значению  
**Зачем:** affinity относится к конкретному candidate item  
**Что получим:** четыре новых признака

In [28]:
validation_ranking_table = validation_ranking_table.merge(
    validation_product_affinity,
    on=["customer_id", "product_type_name"],
    how="left",
).merge(
    validation_colour_affinity,
    on=["customer_id", "colour_group_name"],
    how="left",
).merge(
    validation_section_affinity,
    on=["customer_id", "section_name"],
    how="left",
).merge(
    validation_garment_affinity,
    on=["customer_id", "garment_group_name"],
    how="left",
)
display(validation_ranking_table.head())

                                         customer_id  ... user_garment_group_count
0  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...                      2.0
1  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...                      1.0
2  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...                      2.0
3  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...                      NaN
4  0044c0b989ea44be655875ac6092dd7c86ab21b1218faf...  ...                      NaN

[5 rows x 41 columns]


### Positive future pairs

**Что делаем:** создаём отдельную таблицу target=1  
**Зачем:** future используется только на этапе label  
**Что получим:** `validation_positive_pairs`

In [29]:
validation_positive_pairs = (
    validation_future[["customer_id", "article_id"]]
    .drop_duplicates()
    .assign(target=1)
)
print("Future positive pairs:", len(validation_positive_pairs))

Future positive pairs: 23242


### Target merge

**Что делаем:** left-join positives к generated pairs  
**Зачем:** товары вне candidates не создают новые строки  
**Что получим:** столбец `target`

In [30]:
validation_ranking_table = validation_ranking_table.merge(
    validation_positive_pairs,
    on=["customer_id", "article_id"],
    how="left",
)
validation_ranking_table["target"] = (
    validation_ranking_table["target"].fillna(0).astype("int8")
)
print(validation_ranking_table["target"].value_counts())
print("Positive share:", validation_ranking_table["target"].mean())

target
0    444725
1       128
Name: count, dtype: int64
Positive share: 0.00028773549914241334


### Заполнение пропусков

**Что делаем:** categories заменяем Unknown, numeric NaN — нулём  
**Зачем:** CatBoost table должна иметь полную схему  
**Что получим:** таблицу без missing values

In [31]:
validation_ranking_table[ITEM_FEATURE_COLUMNS] = (
    validation_ranking_table[ITEM_FEATURE_COLUMNS].fillna("Unknown").astype(str)
)
validation_numeric_columns = validation_ranking_table.select_dtypes(
    include="number"
).columns
validation_ranking_table[validation_numeric_columns] = (
    validation_ranking_table[validation_numeric_columns].fillna(0)
)
print("Missing cells:", validation_ranking_table.isna().sum().sum())

Missing cells: 0


### Проверки validation table

**Что делаем:** проверяем duplicates, infinity и temporal boundary  
**Зачем:** ошибка должна появиться до сохранения  
**Что получим:** leakage-safe table

In [32]:
validation_duplicates = validation_ranking_table.duplicated(
    ["customer_id", "article_id"]
).sum()
validation_numeric_values = validation_ranking_table.select_dtypes(
    include="number"
).astype("float64").to_numpy()

assert validation_duplicates == 0
assert not np.isinf(validation_numeric_values).any()
assert not validation_ranking_table.isna().any().any()
assert validation_history["t_dat"].max() < validation_cutoff
print("Duplicate pairs:", validation_duplicates)

Duplicate pairs: 0


### Validation Candidate Recall

**Что делаем:** сравниваем candidates с полным future ground truth  
**Зачем:** target share и coverage отвечают на разные вопросы  
**Что получим:** Candidate Recall@250

In [33]:
validation_users = set(validation_ranking_table["customer_id"])
validation_future_evaluation = validation_future[
    validation_future["customer_id"].isin(validation_users)
]
validation_ground_truth = validation_future_evaluation.sort_values(
    "t_dat"
).drop_duplicates(
    ["customer_id", "article_id"]
).groupby("customer_id", sort=False)["article_id"].apply(list).to_dict()
validation_candidate_lists = validation_ranking_table.groupby(
    "customer_id", sort=False
)["article_id"].apply(list).to_dict()
validation_candidate_recall = candidate_recall_at_k(
    validation_ground_truth, validation_candidate_lists, 250
)
print("Candidate Recall@250:", validation_candidate_recall)

Candidate Recall@250: 0.054291666666666655


### Сохранение validation table

**Что делаем:** записываем features и target  
**Зачем:** notebook 09 загрузит файл напрямую  
**Что получим:** `validation_ranking_table.parquet`

In [34]:
validation_ranking_path = PROCESSED_DIR / "validation_ranking_table.parquet"
validation_ranking_table.to_parquet(validation_ranking_path, index=False)
print("Сохранено:", validation_ranking_path)

Сохранено: <PROJECT_ROOT>/data/processed/validation_ranking_table.parquet


### User feature helper

**Что делаем:** фиксируем уже показанные user aggregations  
**Зачем:** функция не добавляет item, pair или target  
**Что получим:** `build_user_features`

In [35]:
def build_user_features(history, reference_date, customers):
    result = history.groupby("customer_id", as_index=False).agg(
        user_total_purchases=("article_id", "size"),
        user_unique_items=("article_id", "nunique"),
        user_active_days=("t_dat", "nunique"),
        user_last_purchase=("t_dat", "max"),
        user_average_price=("price", "mean"),
    )
    result["user_days_since_last_purchase"] = (
        reference_date - result.pop("user_last_purchase")
    ).dt.days
    online = (
        history.assign(_online=history["sales_channel_id"].eq(2).astype(float))
        .groupby("customer_id", as_index=False)["_online"].mean()
        .rename(columns={"_online": "user_online_share"})
    )
    ages = customers[["customer_id", "age"]].drop_duplicates("customer_id")
    ages = ages.rename(columns={"age": "user_age"})
    return result.merge(online, on="customer_id", how="left").merge(
        ages, on="customer_id", how="left"
    )

### Item feature helper

**Что делаем:** фиксируем уже показанные item aggregations  
**Зачем:** функция использует только history до reference date  
**Что получим:** `build_item_features`

In [36]:
def build_item_features(history, reference_date):
    result = history.groupby("article_id", as_index=False).agg(
        item_total_purchases=("customer_id", "size"),
        item_unique_customers=("customer_id", "nunique"),
        item_average_price=("price", "mean"),
        item_last_purchase=("t_dat", "max"),
    )
    result["item_days_since_last_purchase"] = (
        reference_date - result.pop("item_last_purchase")
    ).dt.days
    recent_7d = history[history["t_dat"] >= reference_date - pd.Timedelta(days=7)]
    counts_7d = recent_7d.groupby("article_id", as_index=False).size()
    counts_7d = counts_7d.rename(columns={"size": "item_popularity_7d"})
    recent_30d = history[history["t_dat"] >= reference_date - pd.Timedelta(days=30)]
    counts_30d = recent_30d.groupby("article_id", as_index=False).size()
    counts_30d = counts_30d.rename(columns={"size": "item_popularity_30d"})
    return result.merge(counts_7d, on="article_id", how="left").merge(
        counts_30d, on="article_id", how="left"
    )

### Pair feature helper

**Что делаем:** фиксируем count и recency одной пары  
**Зачем:** функция возвращает только user-item признаки  
**Что получим:** `build_user_item_features`

In [37]:
def build_user_item_features(history, reference_date):
    result = history.groupby(
        ["customer_id", "article_id"], as_index=False
    ).agg(
        user_item_purchase_count=("article_id", "size"),
        user_item_last_purchase=("t_dat", "max"),
    )
    result["days_since_user_bought_item"] = (
        reference_date - result.pop("user_item_last_purchase")
    ).dt.days
    result["user_bought_item_before"] = 1
    return result

### Affinity helper

**Что делаем:** повторяем product type пример для четырёх categories  
**Зачем:** функция не выполняет остальные feature merges  
**Что получим:** одну affinity table

In [38]:
def build_affinity_features(history, article_attributes, candidates):
    history_categories = history[["customer_id", "article_id"]].merge(
        article_attributes,
        on="article_id",
        how="left",
    )
    output_names = {
        "product_type_name": "user_product_type_count",
        "colour_group_name": "user_colour_count",
        "section_name": "user_section_count",
        "garment_group_name": "user_garment_group_count",
    }
    result = candidates[["customer_id", "article_id"]].merge(
        article_attributes,
        on="article_id",
        how="left",
    )
    for category, output_name in output_names.items():
        counts = history_categories.groupby(
            ["customer_id", category], as_index=False
        ).size().rename(columns={"size": output_name})
        result = result.merge(counts, on=["customer_id", category], how="left")
    output_columns = ["customer_id", "article_id", *output_names.values()]
    return result[output_columns]

### Train: history, future и candidates

**Что делаем:** загружаем входы train-окна  
**Зачем:** готовые candidates приходят из notebook 07  
**Что получим:** три `train_...` таблицы

In [39]:
train_cutoff = pd.Timestamp(windows["train"]["cutoff_date"])
train_end = pd.Timestamp(windows["train"]["target_end_date"])
train_history = transactions[
    transactions["t_dat"] < train_cutoff
].copy()
train_future = transactions[
    transactions["t_dat"].between(train_cutoff, train_end)
].copy()
train_candidates = pd.read_parquet(
    PROCESSED_DIR / "merged_candidates_train.parquet"
)
assert train_history["t_dat"].max() < train_cutoff
print("History:", train_history.shape)
print("Future:", train_future.shape)
print("Candidates:", train_candidates.shape)

History: (998087, 5)
Future: (13793, 5)
Candidates: (445008, 15)


### Train: user features

**Что делаем:** повторяем уже показанные user aggregations  
**Зачем:** функция возвращает только признаки пользователя  
**Что получим:** `train_user_features`

In [40]:
train_user_features = build_user_features(
    train_history,
    train_cutoff,
    customers,
)
print("User features:", train_user_features.shape)

User features: (443629, 8)


### Train: item features

**Что делаем:** повторяем item counts, recency и popularity windows  
**Зачем:** future не участвует в агрегатах  
**Что получим:** `train_item_features`

In [41]:
train_item_features = build_item_features(
    train_history,
    train_cutoff,
)
print("Item features:", train_item_features.shape)

Item features: (49712, 7)


### Train: user-item features

**Что делаем:** повторяем count и last purchase пары  
**Зачем:** эти признаки описывают конкретного кандидата  
**Что получим:** `train_pair_features`

In [42]:
train_pair_features = build_user_item_features(
    train_history,
    train_cutoff,
)
print("Pair features:", train_pair_features.shape)

Pair features: (985415, 5)


### Train: category affinity

**Что делаем:** считаем четыре уже объяснённых category counts  
**Зачем:** один пример product type был показан вручную  
**Что получим:** `train_affinity_features`

In [43]:
train_affinity_features = build_affinity_features(
    train_history,
    article_attributes,
    train_candidates,
)
print("Affinity features:", train_affinity_features.shape)

Affinity features: (445008, 6)


### Train: user и item merge

**Что делаем:** присоединяем первые два блока к candidate pairs  
**Зачем:** каждая строка по-прежнему является одной user-item pair  
**Что получим:** начало `train_ranking_table`

In [44]:
train_ranking_table = train_candidates.merge(
    train_user_features,
    on="customer_id",
    how="left",
)
train_ranking_table = train_ranking_table.merge(
    train_item_features,
    on="article_id",
    how="left",
)
print(train_ranking_table.shape)

(445008, 28)


### Train: pair и category merge

**Что делаем:** добавляем article attributes, pair history и affinity  
**Зачем:** target пока не присоединяется  
**Что получим:** полную feature table без label

In [45]:
train_ranking_table = train_ranking_table.merge(
    article_attributes,
    on="article_id",
    how="left",
)
train_ranking_table = train_ranking_table.merge(
    train_pair_features,
    on=["customer_id", "article_id"],
    how="left",
)
train_ranking_table = train_ranking_table.merge(
    train_affinity_features,
    on=["customer_id", "article_id"],
    how="left",
)
print(train_ranking_table.shape)

(445008, 41)


### Train: positive pairs

**Что делаем:** создаём target=1 только из соответствующей future-недели  
**Зачем:** label не влияет на features  
**Что получим:** `train_positive_pairs`

In [46]:
train_positive_pairs = (
    train_future[["customer_id", "article_id"]]
    .drop_duplicates()
    .assign(target=1)
)
print("Future positive pairs:", len(train_positive_pairs))

Future positive pairs: 13653


### Train: target merge

**Что делаем:** left-join positives к generated candidates  
**Зачем:** непокупка generated pair получает target=0  
**Что получим:** labelled `train_ranking_table`

In [47]:
train_ranking_table = train_ranking_table.merge(
    train_positive_pairs,
    on=["customer_id", "article_id"],
    how="left",
)
train_ranking_table["target"] = (
    train_ranking_table["target"].fillna(0).astype("int8")
)
print(train_ranking_table["target"].value_counts())

target
0    444854
1       154
Name: count, dtype: int64


### Train: пропуски и проверки

**Что делаем:** заполняем categories и numeric NaN, затем проверяем пары  
**Зачем:** CatBoost table не должна содержать missing или duplicates  
**Что получим:** готовую `train_ranking_table`

In [48]:
train_ranking_table[ITEM_FEATURE_COLUMNS] = (
    train_ranking_table[ITEM_FEATURE_COLUMNS].fillna("Unknown").astype(str)
)
train_numeric_columns = train_ranking_table.select_dtypes(
    include="number"
).columns
train_ranking_table[train_numeric_columns] = (
    train_ranking_table[train_numeric_columns].fillna(0)
)
assert not train_ranking_table.duplicated(
    ["customer_id", "article_id"]
).any()
assert not train_ranking_table.isna().any().any()
print("Positive share:", train_ranking_table["target"].mean())

Positive share: 0.0003460611944054938


### Train: сохранение ranking table

**Что делаем:** записываем готовые features и target  
**Зачем:** notebook 09 только загрузит эти таблицы  
**Что получим:** `train_ranking_table.parquet`

In [49]:
train_ranking_path = PROCESSED_DIR / "train_ranking_table.parquet"
train_ranking_table.to_parquet(train_ranking_path, index=False)
print("Сохранено:", train_ranking_path)

Сохранено: <PROJECT_ROOT>/data/processed/train_ranking_table.parquet


### Test: history, future и candidates

**Что делаем:** загружаем входы test-окна  
**Зачем:** готовые candidates приходят из notebook 07  
**Что получим:** три `test_...` таблицы

In [50]:
test_cutoff = pd.Timestamp(windows["test"]["cutoff_date"])
test_end = pd.Timestamp(windows["test"]["target_end_date"])
test_history = transactions[
    transactions["t_dat"] < test_cutoff
].copy()
test_future = transactions[
    transactions["t_dat"].between(test_cutoff, test_end)
].copy()
test_candidates = pd.read_parquet(
    PROCESSED_DIR / "merged_candidates_test.parquet"
)
assert test_history["t_dat"].max() < test_cutoff
print("History:", test_history.shape)
print("Future:", test_future.shape)
print("Candidates:", test_candidates.shape)

History: (1035285, 5)
Future: (13290, 5)
Candidates: (445123, 15)


### Test: user features

**Что делаем:** повторяем уже показанные user aggregations  
**Зачем:** функция возвращает только признаки пользователя  
**Что получим:** `test_user_features`

In [51]:
test_user_features = build_user_features(
    test_history,
    test_cutoff,
    customers,
)
print("User features:", test_user_features.shape)

User features: (454441, 8)


### Test: item features

**Что делаем:** повторяем item counts, recency и popularity windows  
**Зачем:** future не участвует в агрегатах  
**Что получим:** `test_item_features`

In [52]:
test_item_features = build_item_features(
    test_history,
    test_cutoff,
)
print("Item features:", test_item_features.shape)

Item features: (50928, 7)


### Test: user-item features

**Что делаем:** повторяем count и last purchase пары  
**Зачем:** эти признаки описывают конкретного кандидата  
**Что получим:** `test_pair_features`

In [53]:
test_pair_features = build_user_item_features(
    test_history,
    test_cutoff,
)
print("Pair features:", test_pair_features.shape)

Pair features: (1022169, 5)


### Test: category affinity

**Что делаем:** считаем четыре уже объяснённых category counts  
**Зачем:** один пример product type был показан вручную  
**Что получим:** `test_affinity_features`

In [54]:
test_affinity_features = build_affinity_features(
    test_history,
    article_attributes,
    test_candidates,
)
print("Affinity features:", test_affinity_features.shape)

Affinity features: (445123, 6)


### Test: user и item merge

**Что делаем:** присоединяем первые два блока к candidate pairs  
**Зачем:** каждая строка по-прежнему является одной user-item pair  
**Что получим:** начало `test_ranking_table`

In [55]:
test_ranking_table = test_candidates.merge(
    test_user_features,
    on="customer_id",
    how="left",
)
test_ranking_table = test_ranking_table.merge(
    test_item_features,
    on="article_id",
    how="left",
)
print(test_ranking_table.shape)

(445123, 28)


### Test: pair и category merge

**Что делаем:** добавляем article attributes, pair history и affinity  
**Зачем:** target пока не присоединяется  
**Что получим:** полную feature table без label

In [56]:
test_ranking_table = test_ranking_table.merge(
    article_attributes,
    on="article_id",
    how="left",
)
test_ranking_table = test_ranking_table.merge(
    test_pair_features,
    on=["customer_id", "article_id"],
    how="left",
)
test_ranking_table = test_ranking_table.merge(
    test_affinity_features,
    on=["customer_id", "article_id"],
    how="left",
)
print(test_ranking_table.shape)

(445123, 41)


### Test: positive pairs

**Что делаем:** создаём target=1 только из соответствующей future-недели  
**Зачем:** label не влияет на features  
**Что получим:** `test_positive_pairs`

In [57]:
test_positive_pairs = (
    test_future[["customer_id", "article_id"]]
    .drop_duplicates()
    .assign(target=1)
)
print("Future positive pairs:", len(test_positive_pairs))

Future positive pairs: 13175


### Test: target merge

**Что делаем:** left-join positives к generated candidates  
**Зачем:** непокупка generated pair получает target=0  
**Что получим:** labelled `test_ranking_table`

In [58]:
test_ranking_table = test_ranking_table.merge(
    test_positive_pairs,
    on=["customer_id", "article_id"],
    how="left",
)
test_ranking_table["target"] = (
    test_ranking_table["target"].fillna(0).astype("int8")
)
print(test_ranking_table["target"].value_counts())

target
0    444997
1       126
Name: count, dtype: int64


### Test: пропуски и проверки

**Что делаем:** заполняем categories и numeric NaN, затем проверяем пары  
**Зачем:** CatBoost table не должна содержать missing или duplicates  
**Что получим:** готовую `test_ranking_table`

In [59]:
test_ranking_table[ITEM_FEATURE_COLUMNS] = (
    test_ranking_table[ITEM_FEATURE_COLUMNS].fillna("Unknown").astype(str)
)
test_numeric_columns = test_ranking_table.select_dtypes(
    include="number"
).columns
test_ranking_table[test_numeric_columns] = (
    test_ranking_table[test_numeric_columns].fillna(0)
)
assert not test_ranking_table.duplicated(
    ["customer_id", "article_id"]
).any()
assert not test_ranking_table.isna().any().any()
print("Positive share:", test_ranking_table["target"].mean())

Positive share: 0.00028306782619635473


### Test: сохранение ranking table

**Что делаем:** записываем готовые features и target  
**Зачем:** notebook 09 только загрузит эти таблицы  
**Что получим:** `test_ranking_table.parquet`

In [60]:
test_ranking_path = PROCESSED_DIR / "test_ranking_table.parquet"
test_ranking_table.to_parquet(test_ranking_path, index=False)
print("Сохранено:", test_ranking_path)

Сохранено: <PROJECT_ROOT>/data/processed/test_ranking_table.parquet
